In [3]:
# ============================================================
# SETUP — à exécuter en premier dans Colab
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/siap'

# Charger les données brutes sauvegardées au script 1
# (si tu ne les as pas encore sauvegardées, ajoute cette ligne au script 1 :
#  freq.to_csv(f'{BASE}/data/freq_raw.csv', index=False)
#  sev.to_csv(f'{BASE}/data/sev_raw.csv', index=False)  )

freq = pd.read_csv(f'{BASE}/data/freq_raw.csv')
sev  = pd.read_csv(f'{BASE}/data/sev_raw.csv')

print(f"freq : {freq.shape} | sev : {sev.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
freq : (678013, 12) | sev : (26639, 2)


In [4]:
# ============================================================
# BLOC 1 — NETTOYAGE DES ANOMALIES
# ============================================================
# Règle actuarielle : on ne supprime pas, on plafonne (winsorisation)
# Supprimer des sinistres extrêmes = biais dans la prime pure

# 1a. ClaimNb : plafonner à 4 (au-delà c'est souvent une erreur de saisie)
freq['ClaimNb'] = freq['ClaimNb'].clip(upper=4)

# 1b. Exposure : doit être dans ]0, 1]
# Les polices avec exposition quasi-nulle faussent la fréquence
freq = freq[freq['Exposure'] > 0.01].copy()
print(f"Polices après filtre exposition : {len(freq)}")

# 1c. ClaimAmount : winsorisation au 99e percentile
p99 = sev['ClaimAmount'].quantile(0.99)
sev['ClaimAmount_raw'] = sev['ClaimAmount'].copy()   # on garde l'original
sev['ClaimAmount'] = sev['ClaimAmount'].clip(upper=p99)
print(f"Seuil winsorisation (p99) : {p99:.0f} €")
# Typiquement autour de 20 000 €
# Les sinistres au-dessus sont réels mais leur modélisation nécessite
# une approche séparée (réassurance, queue de distribution)

Polices après filtre exposition : 664410
Seuil winsorisation (p99) : 16794 €


In [5]:
# ============================================================
# BLOC 2 — FEATURE ENGINEERING ACTUARIEL
# ============================================================
# L'idée : créer des variables qui ont un sens métier
# Un GLM ne capture pas les non-linéarités → on les crée manuellement
# XGBoost les trouvera seul, mais les avoir aide aussi le ML

# 2a. Tranches d'âge conducteur — effet en U bien connu en actuariat
# Jeunes (<25) et seniors (>70) ont des fréquences plus élevées
def age_band(age):
    if age < 25:   return '1_young'      # fréquence très élevée
    elif age < 35: return '2_adult'
    elif age < 50: return '3_experienced'  # fréquence la plus basse
    elif age < 65: return '4_mature'
    else:          return '5_senior'      # fréquence remonte

freq['DrivAge_band'] = freq['DrivAge'].apply(age_band)

# 2b. Tranches BonusMalus — la variable la plus prédictive
# BonusMalus=50 → meilleur profil (bonus maximum en France)
# BonusMalus=100 → profil neutre (ni bonus ni malus)
# BonusMalus>100 → malussé (accidents récents)
def bm_band(bm):
    if bm <= 60:    return '1_excellent'
    elif bm <= 80:  return '2_good'
    elif bm <= 100: return '3_neutral'
    elif bm <= 130: return '4_bad'
    else:           return '5_very_bad'

freq['BonusMalus_band'] = freq['BonusMalus'].apply(bm_band)

# 2c. Âge véhicule — non-linéarité : véhicules très récents
# et très anciens ont des comportements différents
freq['VehAge_band'] = pd.cut(
    freq['VehAge'],
    bins=[-1, 1, 5, 10, 15, 100],
    labels=['0_new','1_recent','2_middle','3_old','4_very_old']
)

# 2d. Interaction DrivAge × BonusMalus
# Un jeune avec BM=50 est très différent d'un jeune avec BM=150
# C'est une interaction que le GLM de base ne capture pas
freq['young_and_bad'] = (
    (freq['DrivAge'] < 25) & (freq['BonusMalus'] > 100)
).astype(int)

# 2e. Log de la densité de population
# La densité varie de 1 à 27000 → très asymétrique
# Le log la normalise et capture mieux l'effet urbain/rural
freq['log_Density'] = np.log1p(freq['Density'])

print("Features créées :")
print(freq[['DrivAge_band','BonusMalus_band','VehAge_band',
            'young_and_bad','log_Density']].head(3))

Features créées :
  DrivAge_band BonusMalus_band VehAge_band  young_and_bad  log_Density
0     4_mature     1_excellent       0_new              0     7.104965
1     4_mature     1_excellent       0_new              0     7.104965
2     4_mature     1_excellent    1_recent              0     4.007333


In [6]:
# ============================================================
# BLOC 3 — ENCODAGE DES VARIABLES CATÉGORIELLES
# ============================================================
# Pour le GLM : on utilise get_dummies (one-hot encoding)
# Pour XGBoost : il gère nativement les catégories, mais on
#                encode quand même pour uniformité

# Variables catégorielles du dataset
cat_cols = ['Area', 'VehBrand', 'VehGas', 'Region',
            'DrivAge_band', 'BonusMalus_band', 'VehAge_band']

# One-hot encoding — drop_first=True pour éviter la multicolinéarité
# (nécessaire pour le GLM, sans effet pour XGBoost)
freq_encoded = pd.get_dummies(freq, columns=cat_cols, drop_first=True, dtype=int)

print(f"Colonnes après encodage : {freq_encoded.shape[1]}")
# Typiquement ~60–70 colonnes

Colonnes après encodage : 59


In [7]:
# ============================================================
# BLOC 4 — SPLIT TRAIN / TEST ACTUARIEL
# ============================================================
# Important : on NE mélange pas aléatoirement en actuariat
# On simule un split temporel même si les dates ne sont pas disponibles
# → les 80% premières polices = train, les 20% dernières = test
# Raison : en production, le modèle est entraîné sur le passé
#          et prédit sur le futur

from sklearn.model_selection import train_test_split

# On split en gardant la même distribution de ClaimNb
# stratify sur ClaimNb > 0 (sinistre ou pas) pour équilibrer
freq_train, freq_test = train_test_split(
    freq_encoded,
    test_size=0.2,
    random_state=42,
    stratify=(freq_encoded['ClaimNb'] > 0).astype(int)
)

print(f"Train : {len(freq_train)} polices")
print(f"Test  : {len(freq_test)} polices")

# Vérification : la fréquence doit être similaire dans les deux sets
freq_train_rate = freq_train['ClaimNb'].sum() / freq_train['Exposure'].sum()
freq_test_rate  = freq_test['ClaimNb'].sum()  / freq_test['Exposure'].sum()
print(f"Fréquence train : {freq_train_rate:.4f}")
print(f"Fréquence test  : {freq_test_rate:.4f}")
# Si les deux sont proches → le split est bon

Train : 531528 polices
Test  : 132882 polices
Fréquence train : 0.0995
Fréquence test  : 0.0998


In [8]:
# ============================================================
# BLOC 5 — SAUVEGARDE
# ============================================================
import os
os.makedirs(f'{BASE}/data', exist_ok=True)

freq_train.to_csv(f'{BASE}/data/freq_train.csv', index=False)
freq_test.to_csv(f'{BASE}/data/freq_test.csv',  index=False)
sev.to_csv(f'{BASE}/data/sev_clean.csv',        index=False)

# On sauvegarde aussi la liste des features pour le GLM
# (toutes les colonnes sauf les cibles et identifiants)
exclude = ['IDpol', 'ClaimNb', 'Exposure']
feature_cols = [c for c in freq_train.columns if c not in exclude]

import json
with open(f'{BASE}/data/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print(f"Features sauvegardées : {len(feature_cols)} colonnes")
print("Script 2 terminé.")

Features sauvegardées : 56 colonnes
Script 2 terminé.
